## QWEN OMNI

In [1]:
import base64
import json
import os
import traceback
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(".env")

BASE_URL = os.getenv(
    "DASHSCOPE_BASE_URL",
    "https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
)
MODEL = os.getenv("DASHSCOPE_MODEL", "qwen3.5-omni-flash")
API_KEY = os.getenv("DASHSCOPE_API_KEY")

TEST_DATA_DIR = Path("test_data")
OUTPUT_PATH = Path("outputs/test_api_predictions.jsonl")

# Set to None to run every .wav/.txt pair in TEST_DATA_DIR.
# Example: SAMPLE_NAME = "happy"
SAMPLE_NAME = None


In [2]:
def load_audio_text_pairs(data_dir: Path) -> list[dict]:
    samples = []

    for audio_path in sorted(data_dir.glob("*.wav")):
        transcript_path = audio_path.with_suffix(".txt")
        if not transcript_path.exists():
            print(f"Skipping {audio_path.name}: missing {transcript_path.name}")
            continue

        samples.append(
            {
                "sample_name": audio_path.stem,
                "audio_path": audio_path,
                "transcript_path": transcript_path,
                "transcript": transcript_path.read_text(encoding="utf-8").strip(),
            }
        )

    if SAMPLE_NAME is not None:
        samples = [sample for sample in samples if sample["sample_name"] == SAMPLE_NAME]

    if not samples:
        raise RuntimeError(f"No matching .wav/.txt pairs found in {data_dir}")

    return samples


samples = load_audio_text_pairs(TEST_DATA_DIR)
for sample in samples:
    print(f"{sample['sample_name']}: {sample['audio_path']} + {sample['transcript_path']}")


angry: test_data/angry.wav + test_data/angry.txt
happy: test_data/happy.wav + test_data/happy.txt
neutral: test_data/neutral.wav + test_data/neutral.txt
sad: test_data/sad.wav + test_data/sad.txt


In [3]:
def encode_audio_data_url(audio_path: Path) -> tuple[str, str]:
    suffix = audio_path.suffix.lower().lstrip(".")
    if not suffix:
        raise ValueError("Audio file must have an extension, for example .wav or .mp3")

    with audio_path.open("rb") as audio_file:
        encoded = base64.b64encode(audio_file.read()).decode("utf-8")

    return f"data:;base64,{encoded}", suffix


def build_prompt(transcript: str) -> str:
    return f"""
You are analyzing a speech utterance.

Input transcript/context:
{transcript}

Use both the audio and transcript. Return text only.
If this is for speech emotion recognition, choose one label from:
neutral, happy, sad, angry.
Also give a brief explanation based on acoustic and semantic evidence.
""".strip()


def call_model(client: OpenAI, sample: dict) -> tuple[str, object]:
    audio_data_url, audio_format = encode_audio_data_url(sample["audio_path"])
    prompt = build_prompt(sample["transcript"])

    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_audio",
                        "input_audio": {
                            "data": audio_data_url,
                            "format": audio_format,
                        },
                    },
                    {"type": "text", "text": prompt},
                ],
            }
        ],
        modalities=["text"],
        stream=True,
        stream_options={"include_usage": True},
    )

    final_text = ""
    usage = None

    for chunk in completion:
        if chunk.choices and chunk.choices[0].delta.content:
            text = chunk.choices[0].delta.content
            final_text += text
            print(text, end="")

        if getattr(chunk, "usage", None):
            usage = chunk.usage

    return final_text.strip(), usage


In [4]:
if not API_KEY:
    raise RuntimeError("Set DASHSCOPE_API_KEY in .env or in the notebook environment.")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

results = []

try:
    for sample in samples:
        print(f"\n=== {sample['sample_name']} ===")
        prediction_text, usage = call_model(client, sample)
        print("\n")

        result = {
            "sample_name": sample["sample_name"],
            "audio_path": str(sample["audio_path"]),
            "transcript_path": str(sample["transcript_path"]),
            "transcript": sample["transcript"],
            "prediction_text": prediction_text,
            "usage": usage.model_dump() if hasattr(usage, "model_dump") else None,
        }
        results.append(result)

    with OUTPUT_PATH.open("w", encoding="utf-8") as output_file:
        for result in results:
            output_file.write(json.dumps(result, ensure_ascii=False) + "\n")

    print(f"Saved {len(results)} result(s) to {OUTPUT_PATH}")

except Exception as exc:
    print(type(exc))
    print(repr(exc))
    traceback.print_exc()



=== angry ===
angry

The speaker’s tone is sharp and exasperated, especially with the rhetorical “Pass what up?” followed by a rapid, frustrated recounting of the fish’s fate. The semantic content — describing a cycle of struggle and death — combined with the rising intonation and clipped delivery, conveys irritation and anger rather than sadness or happiness. Acoustically, the voice is tense and slightly elevated in pitch, reinforcing the emotional charge of frustration.


=== happy ===
neutral

The tone of the speaker is calm and matter-of-fact, with no strong emotional inflection. The phrase "Think about that" is delivered as a simple prompt for reflection, not with excitement, sadness, or anger. Acoustically, the pitch and pace are steady, supporting a neutral emotional classification.


=== neutral ===
neutral

The speaker’s tone is matter-of-fact and conversational, with no strong emotional inflection. Semantically, they are discussing logistical planning (number of attendees, b

# GEMINI 3.5 flash

In [1]:
# Gemini text-to-text test. This cell is independent from the Omni cells above.
import os
import traceback

from dotenv import load_dotenv

load_dotenv(".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")
print(GEMINI_MODEL)
# Write your own prompt here.
GEMINI_PROMPT = """
hello
""".strip()

if not GEMINI_API_KEY:
    raise RuntimeError("Set GEMINI_API_KEY in .env or in the notebook environment.")

try:
    try:
        from google import genai
    except ImportError as import_error:
        raise ImportError("Install the Gemini SDK first: pip install google-genai") from import_error

    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=GEMINI_PROMPT,
    )

    print(response.text)

except Exception as exc:
    print(type(exc))
    print(repr(exc))
    traceback.print_exc()


gemini-3-flash-preview
Hello! How can I help you today?
